# CHASM Usage Guide

In [2]:
from pathlib import Path

# Define data directories
BASE_DIR = Path("./CHASM_data")
SWPC_SYNOPTIC_DRAWINGS_DIR = BASE_DIR / "swpc_synoptic_drawings"
SAM_SEGMENTATION_MASKS_DIR = BASE_DIR / "sam_segmentation_masks"
SDO_IMAGERY_DIR = BASE_DIR / "sdo_imagery"
CHASM_SELECTIONS_DIR = BASE_DIR / "chasm_selections"
SAM_CHECKPOINTS_DIR = BASE_DIR / "sam_checkpoints"

# Create directories if they don't exist
for directory in [
    SWPC_SYNOPTIC_DRAWINGS_DIR,
    SAM_SEGMENTATION_MASKS_DIR,
    SDO_IMAGERY_DIR,
    CHASM_SELECTIONS_DIR,
    SAM_CHECKPOINTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

## SWPC Synoptic Drawings

In [3]:
# Import the dataset class for the SWPC Synoptic Drawings
from chasm.data import DrawingsDataset

In [4]:
# Get synoptic drawings from our Google Drive
drawings_dataset = DrawingsDataset(root=SWPC_SYNOPTIC_DRAWINGS_DIR, fetch_online=True)

2026-02-10 00:24:35 - chasm.data.auto_download_dataset - INFO: Downloading from https://drive.google.com/uc?id=1HakI-i6iXuqWDFywPx7Ae6RvqIx5tu-0 to CHASM_data\swpc_synoptic_drawings\drawings.tar.gz...
Downloading...
From (original): https://drive.google.com/uc?id=1HakI-i6iXuqWDFywPx7Ae6RvqIx5tu-0
From (redirected): https://drive.google.com/uc?id=1HakI-i6iXuqWDFywPx7Ae6RvqIx5tu-0&confirm=t&uuid=caba7d63-a613-400b-8bde-f7a482e122d5
To: d:\projects\research\CHASM\CHASM_data\swpc_synoptic_drawings\drawings.tar.gz
100%|██████████| 806M/806M [00:14<00:00, 53.8MB/s] 
2026-02-10 00:24:53 - chasm.data.auto_download_dataset - INFO: Extracting CHASM_data\swpc_synoptic_drawings\drawings.tar.gz to CHASM_data\swpc_synoptic_drawings...
2026-02-10 00:25:00 - chasm.data.auto_download_dataset - INFO: Dataset is ready.


In [ ]:
# Get synoptic drawings directly from the SWPC
from chasm.data.generate import SWPCDrawingsScraper

drawings_scraper = SWPCDrawingsScraper(save_path=SWPC_SYNOPTIC_DRAWINGS_DIR)

# Get 1 day
# drawing = drawings_scraper.get_drawing_for_date("2026-01-01")

# Get a list of dates
# dates = ["2025-01-01", "2025-01-15", "2025-02-14"]
# drawings = drawings_scraper.get_drawings_for_dates(dates)

# Get a range of dates
start_date = "2025-01-01"
end_date = "2025-01-31"
drawings = drawings_scraper.get_drawings_for_date_range(start_date=start_date, end_date=end_date)

drawings_dataset = DrawingsDataset(root=SWPC_SYNOPTIC_DRAWINGS_DIR)

## SAM Masks

In [5]:
# Import the dataset class for the SAM segmentation masks
from chasm.data import SAMMaskDataset

In [6]:
# Get SAM segmentation masks from our Google Drive
masks_dataset = SAMMaskDataset(root=SAM_SEGMENTATION_MASKS_DIR, fetch_online=True)

2026-02-10 00:25:13 - chasm.data.auto_download_dataset - INFO: Downloading from https://drive.google.com/uc?id=1Gf5X-o6KPX4IRyWcAqcWBVbMdj4lYc7b to CHASM_data\sam_segmentation_masks\masks.tar.gz...
Downloading...
From (original): https://drive.google.com/uc?id=1Gf5X-o6KPX4IRyWcAqcWBVbMdj4lYc7b
From (redirected): https://drive.google.com/uc?id=1Gf5X-o6KPX4IRyWcAqcWBVbMdj4lYc7b&confirm=t&uuid=225d96e0-36a4-47d4-9a03-07eaa253bae6
To: d:\projects\research\CHASM\CHASM_data\sam_segmentation_masks\masks.tar.gz
100%|██████████| 305M/305M [00:03<00:00, 80.0MB/s] 
2026-02-10 00:25:19 - chasm.data.auto_download_dataset - INFO: Extracting CHASM_data\sam_segmentation_masks\masks.tar.gz to CHASM_data\sam_segmentation_masks...
2026-02-10 00:25:28 - chasm.data.auto_download_dataset - INFO: Dataset is ready.


In [ ]:
# Generate your own SAM segmentation masks
from chasm.data.generate import SAMSegmentationMasksGenerator

# Download a SAM checkpoint from https://github.com/facebookresearch/segment-anything?tab=readme-ov-file#model-checkpoints and put the file in the SAM_CHECKPOINTS_DIR
# vit_h is the largest, best version of SAM
# vit_b will work better on local systems

sam_mask_generator = SAMSegmentationMasksGenerator(
    swpc_drawing_dir=SWPC_SYNOPTIC_DRAWINGS_DIR,
    save_dir=SAM_SEGMENTATION_MASKS_DIR,
    model_type="vit_b",
    sam_checkpoint_filepath=SAM_CHECKPOINTS_DIR / "sam_vit_b_01ec64.pth",
)
# Create SAM segmentation masks for all drawings in the SWPC_SYNOPTIC_DRAWINGS_DIR and save them to SAM_SEGMENTATION_MASKS_DIR
sam_mask_generator.process_all_images()

masks_dataset = SAMMaskDataset(root=SAM_SEGMENTATION_MASKS_DIR)

## SDO Imagery

In [7]:
# Import the dataset class for the SDO imagery
from chasm.data import SDODataset

In [ ]:
# Get SDO imagery from our Google Drive
sdo_dataset = SDODataset(root=SDO_IMAGERY_DIR, fetch_online=True)

In [8]:
# Download SDO imagery on your own
from chasm.data.generate import CHASM_AIADownloader, JSOCQuery
from datetime import datetime

FULL_SIZE_IMAGE_DIR = SDO_IMAGERY_DIR / "full_size_images"
POST_PROCESSED_IMAGE_DIR = SDO_IMAGERY_DIR / "post_processed_images"

sdo_downloader = CHASM_AIADownloader(
    full_save_path=FULL_SIZE_IMAGE_DIR,
    resampled_save_path=POST_PROCESSED_IMAGE_DIR,
    email="cbeckdevelopment@gmail.com"
)

swpc_dates = sdo_downloader.get_dates_from_swpc_drawing_dir(SWPC_SYNOPTIC_DRAWINGS_DIR)
jsoc_queries = [
    sdo_downloader.get_jsoc_query(date, wl)
    for date in swpc_dates
    for wl in JSOCQuery.VALID_WAVELENGTHS
]

def download_progress(completed, total):
    print(f"  Download progress: {completed}/{total}")


download_results = sdo_downloader.download_images_parallel(
    jsoc_queries,
    threshold_minutes=60,
    max_workers=6,
    progress_callback=download_progress,
)

# Post-process the downloaded images
years = [year for year in FULL_SIZE_IMAGE_DIR.iterdir() if year.is_dir()]
print(years)

pairs_to_process = []
for year in years:
    year_dir = FULL_SIZE_IMAGE_DIR / year
    for wl_dir in year_dir.iterdir():
        if wl_dir.is_dir():
            for image in wl_dir.glob("*.fits"):
                image_date = datetime.fromisoformat(image.stem)
                _, resampled_path = sdo_downloader._paths_for_time_and_wavelength(image_date, wl_dir.name)
                resampled_path.parent.mkdir(parents=True, exist_ok=True)
                pairs_to_process.append((image, resampled_path))

if pairs_to_process:
    def postprocess_progress(completed, total):
        print(f"  Post-process progress: {completed}/{total}")

    postprocess_results = sdo_downloader.post_process_parallel(
        pairs_to_process,
        resolution=512,
        max_workers=6,
        progress_callback=postprocess_progress,
    )

sdo_dataset = SDODataset(root=SDO_IMAGERY_DIR)

2026-02-10 00:26:09 - chasm.data.auto_download_dataset - INFO: Dataset already exists at CHASM_data\sdo_imagery, skipping download.
2026-02-10 00:26:09 - chasm.data.auto_download_dataset - INFO: Dataset is ready.


[]


RuntimeError: No data files found in CHASM_data\sdo_imagery

## CHASM

In [ ]:
from chasm.gui.app import run_app

run_app(
    drawings_dataset=drawings_dataset,
    masks_dataset=masks_dataset,
    save_dir=CHASM_SELECTIONS_DIR,
)

## Post-Process CHASM Selections

## Train/Test CHRONNOS with CHASM Selections